In [0]:
# ============================================================
# Silver — Source 10: Partner S3 Affiliate Sales
#
# Transformations:
#   - Cast sale_date, created_at, updated_at to timestamp
#   - Normalise sale_status, partner_type, channel
#   - Validate sale_amount_gbp > 0
#   - Reject null sale_id or product_sku → quarantine
#   - channel null is valid
#   - Deduplicate on sale_id
#
# Source:  bronze.src_10_affiliates.sales
# Target:  silver.src_10_affiliates.sales
# Quarantine: silver.quarantine.src_10_affiliates
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window

BRONZE_CATALOG = 'bronze'
SILVER_CATALOG = 'silver'
TARGET_TABLE = f'{SILVER_CATALOG}.src_10_affiliates.sales'
QUARANTINE_TABLE = f'{SILVER_CATALOG}.quarantine.src_10_affiliates'

VALID_STATUSES = ['completed', 'pending', 'cancelled', 'refunded', 'rejected']

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.src_10_affiliates')
print('Silver Source 10 Affiliates — starting...')


In [0]:
bronze = spark.table(f'{BRONZE_CATALOG}.src_10_affiliates.sales')
total = bronze.count()
print(f'Bronze rows: {total}')

# Cast timestamps
df = bronze \
    .withColumn('sale_date',  F.to_timestamp(F.col('sale_date'))) \
    .withColumn('created_at', F.to_timestamp(F.col('created_at'))) \
    .withColumn('updated_at', F.to_timestamp(F.col('updated_at')))

# Normalise
df = df \
    .withColumn('sale_status',   F.lower(F.trim(F.col('sale_status')))) \
    .withColumn('partner_type',  F.lower(F.trim(F.col('partner_type')))) \
    .withColumn('channel',       F.lower(F.trim(F.col('channel')))) \
    .withColumn('currency',      F.upper(F.trim(F.col('currency')))) \
    .withColumn('product_sku',   F.upper(F.trim(F.col('product_sku'))))

# Bad rows
bad = df.filter(
    F.col('sale_id').isNull() |
    F.col('product_sku').isNull() |
    F.col('sale_amount_gbp').isNull() |
    (F.col('sale_amount_gbp') <= 0) |
    ~F.col('sale_status').isin(VALID_STATUSES)
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('sales'))

# Good rows
good = df.filter(
    F.col('sale_id').isNotNull() &
    F.col('product_sku').isNotNull() &
    F.col('sale_amount_gbp').isNotNull() &
    (F.col('sale_amount_gbp') > 0) &
    F.col('sale_status').isin(VALID_STATUSES)
)

w = Window.partitionBy('sale_id').orderBy(F.col('updated_at').desc())
good = good.withColumn('_rn', F.row_number().over(w)) \
           .filter(F.col('_rn') == 1).drop('_rn')

bad_count = bad.count()
good_count = good.count()
print(f'Affiliate sales: {total} total → {good_count} clean, {bad_count} quarantined ({bad_count/total*100:.1f}%)')

# Write
if spark.catalog.tableExists(TARGET_TABLE):
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    dt.alias('t').merge(good.alias('s'), 't.sale_id = s.sale_id') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    good.write.format('delta').mode('overwrite').saveAsTable(TARGET_TABLE)
print('✅ Written')

# Quarantine
if bad_count > 0:
    bad.select(
        F.lit('src_10_affiliates').alias('source'),
        F.col('source_table'),
        F.col('quarantine_reason'),
        F.current_timestamp().alias('quarantined_at'),
        F.to_json(F.struct(*[c for c in bad.columns if c not in ['quarantine_reason','source_table']])).alias('raw_record')
    ).write.format('delta').mode('append').option('mergeSchema','true').saveAsTable(QUARANTINE_TABLE)
    print(f'✅ {bad_count} quarantined')


In [0]:
count = spark.sql(f'SELECT COUNT(*) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
print(f'silver.src_10_affiliates.sales: {count} rows')
spark.sql(f'SELECT partner_name, sale_status, COUNT(*) as cnt FROM {TARGET_TABLE} GROUP BY partner_name, sale_status ORDER BY cnt DESC').show()
